In [1]:
import os

cache_dir = "../../huggingface_cache"
os.environ['HF_HOME'] = cache_dir

# Create the directory if it doesn't exist
os.makedirs(cache_dir, exist_ok=True)

print(f"✅ Hugging Face cache directory is now set to: {os.environ['HF_HOME']}")

✅ Hugging Face cache directory is now set to: ../../huggingface_cache


In [ ]:
import os
import sys
import subprocess
from huggingface_hub import login

# Add the cloned repository to the Python path
sys.path.append('repository/circuit-tracer')
sys.path.append('repository/circuit-tracer/demos')

# This will prompt you for your token
# login()
login(token="")

In [3]:
"""
Sequential CoT Attribution Analysis
Implementation for analyzing Chain-of-Thought reasoning using attribution graphs
"""
import json
import torch
import numpy as np
from typing import List, Dict, Tuple, Optional, Union, Any
from dataclasses import dataclass
from pathlib import Path
import networkx as nx
from collections import defaultdict

from circuit_tracer import attribute, ReplacementModel
from circuit_tracer.graph import Graph
from circuit_tracer.utils import create_graph_files
from circuit_tracer.graph import prune_graph

from transformers import AutoTokenizer, AutoModelForCausalLM

In [10]:
model_name = 'google/gemma-2-2b-it'
transcoder_name = "gemma"
model = ReplacementModel.from_pretrained(model_name, transcoder_name, dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(model_name)

Fetching 26 files:   0%|          | 0/26 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loaded pretrained model google/gemma-2-2b-it into HookedTransformer


In [12]:
model

ReplacementModel(
  (embed): Embed()
  (hook_embed): HookPoint()
  (blocks): ModuleList(
    (0-25): 26 x TransformerBlock(
      (ln1): RMSNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln1_post): RMSNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2): RMSNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2_post): RMSNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (attn): GroupedQueryAttention(
        (hook_k): HookPoint()
        (hook_q): HookPoint()
        (hook_v): HookPoint()
        (hook_z): HookPoint()
        (hook_attn_scores): HookPoint()
        (hook_pattern): HookPoint()
        (hook_result): HookPoint()
        (hook_rot_k): HookPoint()
        (hook_rot_q): HookPoint()
      )
      (mlp): ReplacementMLP(
        (old_mlp): GatedMLP(
          (hook_pre): HookPoint()
          (hook_pr

In [13]:
prompt = "The capital of state containing Dallas is"
answer = model.generate(prompt, max_new_tokens=10, do_sample=False)
print(answer)

  0%|          | 0/10 [00:00<?, ?it/s]

The capital of state containing Dallas is:

A. Austin
B. Houston



In [17]:
prompt = "James writes a 3-page letter to 2 different friends twice a week. How many pages does he write a year?" #Answer: 624
answer = model.generate(prompt, max_new_tokens=224, do_sample=True)
print(answer)

  0%|          | 0/224 [00:00<?, ?it/s]

James writes a 3-page letter to 2 different friends twice a week. How many pages does he write a year? 

**Solution:**

* **Letters per year:** He writes 2 letters * 2 times a week * 52 weeks/year = 208 letters a year
* **Total pages:** 208 letters * 3 pages/letter = 624 pages per year 


Answer: James writes 624 pages a year. 



In [28]:
prompt = "Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?" #Answer: $10
answer = model.generate(prompt, max_new_tokens=224, do_sample=False)
print(answer)

  0%|          | 0/224 [00:00<?, ?it/s]

Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?

Here's how to solve the problem:

**1. Convert minutes to hours:**

* There are 60 minutes in an hour, so 50 minutes is equal to 50/60 = 5/6 of an hour.

**2. Calculate earnings:**

* Weng earns $12 per hour, so for 5/6 of an hour, she earns $12 * (5/6) = $10.

**Answer:** Weng earned $10 yesterday. 



### Run original Gemma-2b model

In [23]:
model_name_base = 'google/gemma-2-2b'
transcoder_name = "gemma"
model_base = ReplacementModel.from_pretrained(model_name_base, transcoder_name, dtype=torch.bfloat16)

Fetching 26 files:   0%|          | 0/26 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/46.4k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Loaded pretrained model google/gemma-2-2b into HookedTransformer


In [29]:
prompt = "Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn? The correct answer is" #Answer: $10
answer = model_base.generate(prompt, max_new_tokens=224, do_sample=False)
print(answer)

  0%|          | 0/224 [00:00<?, ?it/s]

Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn? The correct answer is $60.

The correct answer is $60.

The correct answer is $60.

The correct answer is $60.

The correct answer is $60.

The correct answer is $60.

The correct answer is $60.

The correct answer is $60.

The correct answer is $60.

The correct answer is $60.

The correct answer is $60.

The correct answer is $60.

The correct answer is $60.

The correct answer is $60.

The correct answer is $60.

The correct answer is $60.

The correct answer is $60.

The correct answer is $60.

The correct answer is $60.

The correct answer is $60.

The correct answer is $60.

The correct answer is $60.

The correct answer is $60.

The correct answer is $60.

The correct answer is $60.

The correct answer


In [24]:
model_base

ReplacementModel(
  (embed): Embed()
  (hook_embed): HookPoint()
  (blocks): ModuleList(
    (0-25): 26 x TransformerBlock(
      (ln1): RMSNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln1_post): RMSNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2): RMSNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2_post): RMSNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (attn): GroupedQueryAttention(
        (hook_k): HookPoint()
        (hook_q): HookPoint()
        (hook_v): HookPoint()
        (hook_z): HookPoint()
        (hook_attn_scores): HookPoint()
        (hook_pattern): HookPoint()
        (hook_result): HookPoint()
        (hook_rot_k): HookPoint()
        (hook_rot_q): HookPoint()
      )
      (mlp): ReplacementMLP(
        (old_mlp): GatedMLP(
          (hook_pre): HookPoint()
          (hook_pr